# Notebook 10: SASRec DLRM Re-Ranking Model

## Purpose

This notebook trains DLRM on the SASRec retrieval pipeline's features. SASRec produces a single 128-dim embedding per user (like Two-Tower), so the feature vector is 105-dim with field_dims=[1, 24, 73, 7].

In [ ]:
import numpy as np
import pandas as pd
import pickle
import time
import json
import os
import gc
from pathlib import Path
from typing import List

os.environ['OMP_NUM_THREADS'] = '1'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
from torch.optim.lr_scheduler import OneCycleLR
import xgboost as xgb
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
from scipy import stats
from scipy.stats import false_discovery_control

DATA_DIR = Path('../data/processed')
MODEL_DIR = Path('../models/sasrec')
MODEL_DIR.mkdir(exist_ok=True)
PLOT_DIR = Path('../plots')
PLOT_DIR.mkdir(exist_ok=True)

device = torch.device('mps' if torch.backends.mps.is_available() else 'cpu')
print(f'Device: {device}')

## Section 1: Load SASRec Data and Build Features

In [ ]:
with open(DATA_DIR / 'metadata.pkl', 'rb') as f:
    metadata = pickle.load(f)
n_users = metadata['n_users']
n_movies = metadata['n_movies']
user2idx = metadata['user2idx']
movie2idx = metadata['movie2idx']

user_embeddings = np.load(MODEL_DIR / 'user_embeddings.npy')  # (n_users, 128)
item_embeddings = np.load(MODEL_DIR / 'item_embeddings.npy')  # (n_movies, 128)

user_features_df = pd.read_parquet(DATA_DIR / 'user_features.parquet')
item_features_df = pd.read_parquet(DATA_DIR / 'item_features.parquet')
user_feat_cols = user_features_df.columns.tolist()
item_feat_cols = item_features_df.columns.tolist()

user_feat_matrix = np.zeros((n_users, len(user_feat_cols)), dtype=np.float32)
for uid, uidx in user2idx.items():
    if uid in user_features_df.index:
        user_feat_matrix[uidx] = user_features_df.loc[uid].values

item_feat_matrix = np.zeros((n_movies, len(item_feat_cols)), dtype=np.float32)
for mid, midx in movie2idx.items():
    if mid in item_features_df.index:
        item_feat_matrix[midx] = item_features_df.loc[mid].values

del user_features_df, item_features_df; gc.collect()
print(f'SASRec embeddings: users={user_embeddings.shape}, items={item_embeddings.shape}')
print(f'Features: user={len(user_feat_cols)}, item={len(item_feat_cols)}, total=105')

### Loading Train/Val/Test Splits and Constructing the 105-Dimensional Feature Matrix

The previous cell loaded raw embeddings and side-feature matrices. This cell takes the next critical step: it loads the pre-split train, validation, and test DataFrames along with their corresponding cross-interaction features, then assembles the full 105-dimensional feature vector for every sample.

**Why this step exists:** DLRM operates on a fixed-width numeric feature vector rather than raw IDs or variable-length sequences. We must materialize all 105 features (1 retrieval score from the dot product of SASRec user/item embeddings, 24 user demographic features, 73 item content features, and 7 hand-crafted cross features) into contiguous NumPy arrays that can later be fed to PyTorch DataLoaders efficiently.

**What the code does:**
1. Loads the parquet-serialized train/val/test interaction tables and their cross-feature companions.
2. Filters out any padding rows (user_idx == 0) from validation and test sets -- these are artifacts of the retrieval stage and carry no signal.
3. Defines `build_features_chunked`, which constructs the 105-dim vector in memory-friendly chunks: for each chunk it computes the SASRec retrieval score (dot product between user and item embeddings), concatenates the pre-computed user features, item features, and cross features.
4. Subsamples training data to approximately 3 million rows by selecting entire user histories (preserving per-user ranking structure) to keep training tractable on a single machine.
5. Builds final `X_train`, `X_val`, `X_test` matrices and frees intermediate DataFrames.

**What to expect from the output:** Print statements reporting the subsampled training size, the shapes of all three feature matrices (each with 105 columns), and wall-clock time for the feature construction. The validation and test matrices are built from the full (unsampled) splits so evaluation remains unbiased.

In [ ]:
train_df = pd.read_parquet(DATA_DIR / 'train_set.parquet')
val_df = pd.read_parquet(DATA_DIR / 'val_set.parquet')
test_df = pd.read_parquet(DATA_DIR / 'test_set.parquet')
train_interaction_feats = pd.read_parquet(DATA_DIR / 'train_interaction_features.parquet')
val_interaction_feats = pd.read_parquet(DATA_DIR / 'val_interaction_features.parquet')
test_interaction_feats = pd.read_parquet(DATA_DIR / 'test_interaction_features.parquet')

valid_val_mask = val_df['user_idx'] > 0
val_df = val_df[valid_val_mask].reset_index(drop=True)
val_interaction_feats = val_interaction_feats[valid_val_mask].reset_index(drop=True)
valid_test_mask = test_df['user_idx'] > 0
test_df = test_df[valid_test_mask].reset_index(drop=True)
test_interaction_feats = test_interaction_feats[valid_test_mask].reset_index(drop=True)

def build_features_chunked(user_idxs, movie_idxs, cross_arr, chunk_size=500_000):
    n = len(user_idxs)
    features = np.empty((n, 105), dtype=np.float32)
    for start in range(0, n, chunk_size):
        end = min(start + chunk_size, n)
        u = user_idxs[start:end]; m = movie_idxs[start:end]
        features[start:end, 0] = np.sum(user_embeddings[u] * item_embeddings[m], axis=1)
        features[start:end, 1:25] = user_feat_matrix[u]
        features[start:end, 25:98] = item_feat_matrix[m]
        features[start:end, 98:] = cross_arr[start:end]
    return features

TRAIN_SAMPLE_SIZE = 3_000_000
np.random.seed(42)
if len(train_df) > TRAIN_SAMPLE_SIZE:
    user_groups = train_df.groupby('user_idx').size()
    users_shuffled = user_groups.index.values.copy()
    np.random.shuffle(users_shuffled)
    cumulative = 0; selected = []
    for u in users_shuffled:
        selected.append(u); cumulative += user_groups[u]
        if cumulative >= TRAIN_SAMPLE_SIZE: break
    train_mask = train_df['user_idx'].isin(set(selected))
    train_user_idxs = train_df.loc[train_mask, 'user_idx'].values.copy()
    train_movie_idxs = train_df.loc[train_mask, 'movie_idx'].values.copy()
    y_train = train_df.loc[train_mask, 'label'].values.astype(np.float32)
    train_cross = train_interaction_feats.loc[train_mask].values.astype(np.float32)
    print(f'Subsampled: {len(y_train):,} rows')
else:
    train_user_idxs = train_df['user_idx'].values.copy()
    train_movie_idxs = train_df['movie_idx'].values.copy()
    y_train = train_df['label'].values.astype(np.float32)
    train_cross = train_interaction_feats.values.astype(np.float32)

del train_df, train_interaction_feats; gc.collect()

print('Building feature matrices...')
t0 = time.time()
X_train = build_features_chunked(train_user_idxs, train_movie_idxs, train_cross)
del train_cross; gc.collect()
print(f'  X_train: {X_train.shape}, {time.time()-t0:.1f}s')

val_user_idxs = val_df['user_idx'].values.copy()
val_movie_idxs = val_df['movie_idx'].values.copy()
y_val = val_df['label'].values.astype(np.float32)
X_val = build_features_chunked(val_user_idxs, val_movie_idxs, val_interaction_feats.values.astype(np.float32))

test_user_idxs = test_df['user_idx'].values.copy()
test_movie_idxs = test_df['movie_idx'].values.copy()
y_test = test_df['label'].values.astype(np.float32)
X_test = build_features_chunked(test_user_idxs, test_movie_idxs, test_interaction_feats.values.astype(np.float32))

del val_df, test_df, val_interaction_feats, test_interaction_feats; gc.collect()
print(f'  X_val: {X_val.shape}, X_test: {X_test.shape}')

## Section 2: DLRM Training (105-dim, field_dims=[1, 24, 73, 7])

In [ ]:
class BottomMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, embed_dim, dropout=0.1):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers.extend([nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)])
            prev = h
        layers.append(nn.Linear(prev, embed_dim))
        self.mlp = nn.Sequential(*layers)
    def forward(self, x): return self.mlp(x)

class FieldProjection(nn.Module):
    def __init__(self, dim, embed_dim):
        super().__init__()
        self.proj = nn.Sequential(nn.BatchNorm1d(dim), nn.Linear(dim, embed_dim), nn.ReLU())
    def forward(self, x): return self.proj(x)

class DotProductInteraction(nn.Module):
    def __init__(self, n_fields):
        super().__init__()
        self.idx = torch.triu_indices(n_fields, n_fields, offset=1)
    def forward(self, embs):
        s = torch.stack(embs, dim=1)
        m = torch.bmm(s, s.transpose(1, 2))
        return m[:, self.idx[0], self.idx[1]]

class TopMLP(nn.Module):
    def __init__(self, input_dim, hidden_dims, dropout=0.2):
        super().__init__()
        layers = []
        prev = input_dim
        for h in hidden_dims:
            layers.extend([nn.Linear(prev, h), nn.BatchNorm1d(h), nn.ReLU(), nn.Dropout(dropout)])
            prev = h
        layers.append(nn.Linear(prev, 1))
        self.mlp = nn.Sequential(*layers)
    def forward(self, x): return self.mlp(x).squeeze(-1)

class DLRM(nn.Module):
    def __init__(self, input_dim=105, field_dims=None, embed_dim=64, bottom_mlp_dims=None, top_mlp_dims=None, dropout=0.1):
        super().__init__()
        if field_dims is None: field_dims = [1, 24, 73, 7]
        if bottom_mlp_dims is None: bottom_mlp_dims = [256, 128]
        if top_mlp_dims is None: top_mlp_dims = [128, 64]
        self.field_dims = field_dims
        self.bounds = [0]
        for d in field_dims: self.bounds.append(self.bounds[-1] + d)
        self.bottom_mlp = BottomMLP(input_dim, bottom_mlp_dims, embed_dim, dropout)
        self.field_projs = nn.ModuleList([FieldProjection(d, embed_dim) for d in field_dims])
        nf = len(field_dims) + 1
        self.interaction = DotProductInteraction(nf)
        ni = nf * (nf - 1) // 2
        self.top_mlp = TopMLP(ni + embed_dim, top_mlp_dims, dropout)
        self._config = {'input_dim': input_dim, 'field_dims': field_dims, 'embed_dim': embed_dim,
                        'bottom_mlp_dims': bottom_mlp_dims, 'top_mlp_dims': top_mlp_dims, 'dropout': dropout}
    def forward(self, x):
        fields = [x[:, self.bounds[i]:self.bounds[i+1]] for i in range(len(self.field_dims))]
        de = self.bottom_mlp(x)
        fe = [p(f) for p, f in zip(self.field_projs, fields)]
        inter = self.interaction([de] + fe)
        return self.top_mlp(torch.cat([inter, de], dim=1))
    def get_config(self): return self._config.copy()

model = DLRM(input_dim=105, field_dims=[1, 24, 73, 7]).to(device)
print(f'DLRM (SASRec): {sum(p.numel() for p in model.parameters()):,} params')

### Training Loop with Hybrid BPR + BCE Loss and Early Stopping

With the DLRM architecture defined and instantiated, we now run the actual training procedure. This is the most computationally intensive cell in the notebook and will take several minutes depending on hardware.

**Why this step exists:** Defining a model architecture does nothing until its parameters are optimized against data. This cell implements the full training loop that learns to rank items by relevance for each user. The choice of loss function is particularly important: we use a 50/50 blend of binary cross-entropy (BCE) and Bayesian Personalized Ranking (BPR) loss. BCE treats each sample independently and optimizes pointwise classification accuracy, while BPR explicitly contrasts positive-negative pairs to learn correct orderings -- combining both yields a model that is both well-calibrated and ranking-aware.

**What the code does:**
1. Defines `compute_bpr_loss`: samples random positive-negative pairs within each batch and computes the log-sigmoid of score differences, encouraging positive items to score higher than negatives.
2. Defines `compute_ndcg`: a NumPy-based NDCG@10 evaluator that groups predictions by user and computes normalized discounted cumulative gain -- the primary ranking quality metric.
3. Creates a PyTorch DataLoader (batch size 4096), AdamW optimizer with weight decay 1e-5, and a OneCycleLR scheduler that warms up over the first 10% of training then anneals.
4. Runs up to 30 epochs with patience-5 early stopping on validation NDCG@10. Each epoch: forward pass, hybrid loss, gradient clipping at norm 1.0, then full validation scoring.
5. Saves the best model checkpoint (by validation NDCG@10) to disk.

**What to expect from the output:** A formatted table showing epoch number, training loss, validation AUC, validation NDCG@10, and wall-clock time per epoch. Training will stop early once NDCG@10 plateaus for 5 consecutive epochs, and the final line reports the best epoch and its validation NDCG@10 score.

In [ ]:
def compute_bpr_loss(scores, labels):
    pm = labels == 1; nm = labels == 0
    if pm.sum() == 0 or nm.sum() == 0: return torch.tensor(0.0, device=scores.device)
    ps, ns = scores[pm], scores[nm]
    n = min(len(ps), len(ns))
    return -F.logsigmoid(ps[torch.randperm(len(ps), device=scores.device)[:n]] - ns[torch.randperm(len(ns), device=scores.device)[:n]]).mean()

def compute_ndcg(scores, labels, user_idxs, k=10, max_users=2000):
    si = np.argsort(user_idxs, kind='stable')
    ss, ls, us = scores[si], labels[si], user_idxs[si]
    uu, starts = np.unique(us, return_index=True)
    ends = np.append(starts[1:], len(us))
    ndcgs = []
    for i in range(min(len(uu), max_users)):
        s, e = starts[i], ends[i]
        if e-s < 2: continue
        gl, gs = ls[s:e], ss[s:e]
        if gl.sum() == 0: continue
        rl = gl[np.argsort(gs)[::-1]]; ak = min(k, len(rl))
        dcg = np.sum(rl[:ak] / np.log2(np.arange(2, ak+2)))
        idcg = np.sum(np.sort(gl)[::-1][:ak] / np.log2(np.arange(2, ak+2)))
        if idcg > 0: ndcgs.append(dcg/idcg)
    return np.mean(ndcgs) if ndcgs else 0.0

BATCH_SIZE = 4096; LR = 1e-3; MAX_EPOCHS = 30; PATIENCE = 5
train_ds = TensorDataset(torch.tensor(X_train, dtype=torch.float32), torch.tensor(y_train, dtype=torch.float32))
train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=0, pin_memory=True)
optimizer = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=1e-5)
scheduler = OneCycleLR(optimizer, max_lr=LR, total_steps=len(train_loader)*MAX_EPOCHS, pct_start=0.1)

print(f'Training DLRM (SASRec, 105-dim)...')
print(f'{"Ep":<5}{"Loss":<10}{"Val AUC":<10}{"NDCG@10":<10}{"Time":<6}')
print('-' * 41)

history = []; best_ndcg = -1; best_ep = -1; pat = 0
for epoch in range(MAX_EPOCHS):
    t0 = time.time(); model.train(); el = 0; nb = 0
    for bX, by in train_loader:
        bX, by = bX.to(device), by.to(device)
        sc = model(bX)
        loss = 0.5*F.binary_cross_entropy_with_logits(sc, by) + 0.5*compute_bpr_loss(sc, by)
        optimizer.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step(); scheduler.step()
        el += loss.item(); nb += 1
    model.eval()
    with torch.no_grad():
        vs = model(torch.tensor(X_val, dtype=torch.float32).to(device)).cpu().numpy()
    va = roc_auc_score(y_val, vs); vn = compute_ndcg(vs, y_val, val_user_idxs)
    history.append({'epoch': epoch+1, 'loss': el/nb, 'val_auc': va, 'val_ndcg': vn, 'time': time.time()-t0})
    print(f'{epoch+1:<5}{el/nb:<10.4f}{va:<10.4f}{vn:<10.4f}{time.time()-t0:<6.1f}s')
    if vn > best_ndcg:
        best_ndcg = vn; best_ep = epoch+1; pat = 0
        torch.save(model.state_dict(), MODEL_DIR / 'dlrm_ranker.pt')
    else:
        pat += 1
        if pat >= PATIENCE:
            print(f'Early stopping. Best: ep {best_ep}, NDCG@10={best_ndcg:.4f}')
            break
print(f'Best epoch: {best_ep}, val NDCG@10: {best_ndcg:.4f}')

### Persisting Model Configuration and Training History

After training completes, we need to serialize not just the model weights (already saved during training as the best checkpoint) but also the architectural hyperparameters and training metadata. This enables reproducibility and downstream comparison without re-running the training loop.

**Why this step exists:** A saved `.pt` file contains only the learned weight tensors -- it does not record how the model was constructed (field dimensions, MLP widths, embedding size) or how training progressed. Without this metadata, loading the model later would require manually specifying the same constructor arguments, and comparing experiments across notebooks would be impossible. Saving the training history also enables post-hoc learning curve analysis.

**What the code does:**
1. Extracts the model's configuration dictionary (input_dim, field_dims, embed_dim, MLP dimensions, dropout) via `get_config()`.
2. Appends training metadata: best epoch number, best validation NDCG@10, and total training set size.
3. Writes the config to `dlrm_config.json` for human-readable inspection and programmatic loading.
4. Pickles the full epoch-by-epoch training history (loss, AUC, NDCG, timing) for later plotting or analysis.

**What to expect from the output:** A confirmation message showing the saved model file path and its size in megabytes, confirming that all artifacts are persisted to the `models/sasrec/` directory.

In [ ]:
config = model.get_config()
config['training'] = {'best_epoch': best_ep, 'best_val_ndcg': float(best_ndcg), 'n_train': len(X_train)}
with open(MODEL_DIR / 'dlrm_config.json', 'w') as f:
    json.dump(config, f, indent=2)
with open(MODEL_DIR / 'dlrm_training_history.pkl', 'wb') as f:
    pickle.dump(history, f)
print(f'Saved: {MODEL_DIR}/dlrm_ranker.pt ({os.path.getsize(MODEL_DIR / "dlrm_ranker.pt")/1e6:.2f} MB)')

## Section 3: Test Evaluation and Statistical Comparison

In [ ]:
model.load_state_dict(torch.load(MODEL_DIR / 'dlrm_ranker.pt', map_location=device, weights_only=True))
model.eval()
with torch.no_grad():
    dlrm_scores = model(torch.tensor(X_test, dtype=torch.float32).to(device)).cpu().numpy()

xgb_model = xgb.Booster()
xgb_model.load_model(str(MODEL_DIR / 'xgboost_ranker.json'))
with open(MODEL_DIR / 'ranker_feature_names.pkl', 'rb') as f:
    fnames = pickle.load(f)
xgb_scores = xgb_model.predict(xgb.DMatrix(X_test, feature_names=fnames))

dlrm_auc = roc_auc_score(y_test, dlrm_scores)
xgb_auc = roc_auc_score(y_test, xgb_scores)
dlrm_ndcg = compute_ndcg(dlrm_scores, y_test, test_user_idxs)
xgb_ndcg = compute_ndcg(xgb_scores, y_test, test_user_idxs)

print(f'SASRec Pipeline Test Results:')
print(f'{"Model":<10}{"AUC":<10}{"NDCG@10":<10}')
print(f'{"XGBoost":<10}{xgb_auc:<10.4f}{xgb_ndcg:<10.4f}')
print(f'{"DLRM":<10}{dlrm_auc:<10.4f}{dlrm_ndcg:<10.4f}')
print(f'{"Delta":<10}{dlrm_auc-xgb_auc:<+10.4f}{dlrm_ndcg-xgb_ndcg:<+10.4f}')

### Comprehensive Per-User Metrics and Paired Statistical Significance Testing

The previous cell gave us aggregate AUC and NDCG@10 point estimates comparing DLRM to XGBoost. However, point estimates alone cannot tell us whether observed differences are real or just noise. This cell performs rigorous statistical hypothesis testing to determine if DLRM's performance is significantly different from XGBoost's.

**Why this step exists:** In recommendation system research, small differences in aggregate metrics can be misleading -- they may arise from a few outlier users or random variation. A proper evaluation requires: (1) computing per-user metrics so each user serves as an independent observation, and (2) applying paired statistical tests that account for the fact that both models score the same users. Additionally, testing multiple metrics simultaneously inflates the false positive rate, so we apply Benjamini-Hochberg FDR correction to control for multiple comparisons.

**What the code does:**
1. Defines `compute_full_metrics`: for each user in the test set, computes NDCG@{5,10,20}, Precision@{5,10,20}, MRR, and per-user AUC -- yielding arrays of per-user scores for each metric.
2. Computes these metric arrays for both DLRM and XGBoost predictions on the same test users.
3. For each of the 8 metrics, runs a paired t-test (since the same users appear in both conditions) to get raw p-values and computes Cohen's d effect size to quantify practical significance.
4. Applies Benjamini-Hochberg FDR correction across all 8 tests to control the expected proportion of false discoveries at the 0.05 level.

**What to expect from the output:** A formatted table with one row per metric showing: the mean difference (DLRM minus XGBoost), raw p-value, FDR-adjusted p-value, Cohen's d effect size, and a Yes/No significance indicator. The final line summarizes how many of the 8 metrics show statistically significant differences after correction. This tells us whether DLRM genuinely improves over (or underperforms relative to) the XGBoost baseline on this pipeline.

In [ ]:
def compute_full_metrics(scores, labels, user_idxs, K_values=[5, 10, 20]):
    si = np.argsort(user_idxs, kind='stable')
    ss, ls, us = scores[si], labels[si], user_idxs[si]
    uu, starts = np.unique(us, return_index=True)
    ends = np.append(starts[1:], len(us))
    metrics = {f'ndcg@{k}': [] for k in K_values}
    metrics.update({f'precision@{k}': [] for k in K_values})
    metrics['mrr'] = []; metrics['auc'] = []
    for i in range(len(uu)):
        s, e = starts[i], ends[i]
        if e-s < 2: continue
        gl, gs = ls[s:e], ss[s:e]
        if gl.sum() == 0 or gl.sum() == len(gl): continue
        rl = gl[np.argsort(gs)[::-1]]
        fp = np.where(rl == 1)[0]
        metrics['mrr'].append(1.0/(fp[0]+1) if len(fp) > 0 else 0.0)
        try: metrics['auc'].append(roc_auc_score(gl, gs))
        except: pass
        for k in K_values:
            ak = min(k, len(rl)); tk = rl[:ak]
            metrics[f'precision@{k}'].append(tk.sum()/k)
            dcg = np.sum(tk/np.log2(np.arange(2,ak+2)))
            idcg = np.sum(np.sort(gl)[::-1][:ak]/np.log2(np.arange(2,ak+2)))
            metrics[f'ndcg@{k}'].append(dcg/idcg if idcg > 0 else 0.0)
    return {k: np.array(v) for k, v in metrics.items()}

dm = compute_full_metrics(dlrm_scores, y_test, test_user_idxs)
xm = compute_full_metrics(xgb_scores, y_test, test_user_idxs)

mnames = ['ndcg@5', 'ndcg@10', 'ndcg@20', 'precision@5', 'precision@10', 'precision@20', 'mrr', 'auc']
raw_pvals = []
results = []
for m in mnames:
    dv, xv = dm[m], xm[m]
    n = min(len(dv), len(xv)); dv, xv = dv[:n], xv[:n]
    _, pval = stats.ttest_rel(dv, xv)
    diff = dv.mean() - xv.mean()
    d = diff / (dv-xv).std(ddof=1) if (dv-xv).std(ddof=1) > 0 else 0
    raw_pvals.append(pval)
    results.append({'metric': m, 'diff': diff, 'pval': pval, 'd': d})

adj_pvals = false_discovery_control(raw_pvals, method='bh')

print(f'\nSASRec DLRM vs XGBoost Statistical Significance:')
print(f'{"Metric":<14}{"Diff":<12}{"p-value":<12}{"FDR-adj":<12}{"Cohen d":<10}{"Sig?":<6}')
print('-' * 66)
for i, r in enumerate(results):
    sig = 'Yes' if adj_pvals[i] < 0.05 else 'No'
    print(f'{r["metric"]:<14}{r["diff"]:<+12.4f}{r["pval"]:<12.2e}{adj_pvals[i]:<12.2e}{r["d"]:<+10.3f}{sig:<6}')
print(f'\nSignificant after FDR: {sum(1 for p in adj_pvals if p < 0.05)}/{len(mnames)}')